# 05 — Model Explainability with SHAP

**Project**: LLM-Based Startup Risk Profiling and Funding Outcome Prediction

This notebook covers:
1. SHAP summary plot (global feature importance)
2. SHAP dependence plots for key features
3. Individual prediction explanation (waterfall)
4. Permutation importance comparison
5. Business interpretation of feature contributions

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

try:
    import shap
    print(f'SHAP version: {shap.__version__}')
except ImportError:
    print('SHAP not available. Install: pip install shap')

In [ ]:
from src.data_loader import load_and_preprocess, load_config
from src.train import split_data

config = load_config()
df = load_and_preprocess(config['dataset']['file'])
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

model = joblib.load('../models/best_model.joblib')
print('Model loaded.')

## 1. Extract Preprocessed Features

In [ ]:
preprocessor = model.named_steps['preprocessor']
classifier = model.named_steps['classifier']

X_test_transformed = preprocessor.transform(X_test)
print(f'Transformed shape: {X_test_transformed.shape}')

## 2. SHAP Summary Plot

In [ ]:
try:
    explainer = shap.TreeExplainer(classifier)
    shap_values = explainer.shap_values(X_test_transformed)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    shap.summary_plot(shap_values, X_test_transformed, max_display=15, show=False)
    plt.title('SHAP Feature Importance — Startup Funding Prediction')
    plt.tight_layout()
    plt.savefig('../reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'SHAP analysis failed: {e}')

## 3. Individual Prediction Explanation

In [ ]:
try:
    idx = 0
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[idx],
            base_values=explainer.expected_value,
            data=X_test_transformed[idx],
            feature_names=[f'f{i}' for i in range(X_test_transformed.shape[1])]
        ),
        max_display=10, show=False
    )
    plt.tight_layout()
    plt.savefig('../reports/figures/shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Waterfall plot failed: {e}')

## 4. Business Interpretation

Key features to interpret:
- **company_age**: Older companies may have proven track record
- **funding_velocity**: Higher velocity may indicate market validation
- **rounds_per_year**: Rapid fundraising may signal momentum
- **avg_round_size**: Larger rounds = stronger investor conviction
- **description_length**: More detailed descriptions may correlate with professionalism
- **high_risk_score**: Text-based risk keywords = elevated failure probability

Note: SHAP shows *association*, not *causation*.